# Phase 2 — Feature Validation

Loads `weekly_features.parquet` produced by the feature engineering notebook and runs
structural, range, persona, and churn-decay assertions.

**Prerequisite**: run `phase2_feature_engineering.ipynb` first.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import datetime

spark = SparkSession.builder \
    .appName("gfn-feature-validation") \
    .master("local[*]") \
    .getOrCreate()

weekly_features = spark.read.parquet("/home/spark/work/data/processed/weekly_features.parquet")
users = spark.read.parquet("/home/spark/work/data/raw/users.parquet")

print(f"weekly_features: {weekly_features.count():,} rows, {len(weekly_features.columns)} cols")
print(f"Columns: {weekly_features.columns}")

## Level 1 — Structural Integrity

In [ ]:
total_users = users.count()
wf_rows = weekly_features.count()
wf_users = weekly_features.select("user_id").distinct().count()

print(f"Total users: {total_users:,}")
print(f"weekly_features rows: {wf_rows:,}")
print(f"Unique users in weekly_features: {wf_users:,}")

# Every user should appear exactly 4 times (weeks 1-4)
assert wf_rows == total_users * 4, f"Expected {total_users * 4} rows, got {wf_rows}"
assert wf_users == total_users, f"Expected {total_users} unique users, got {wf_users}"
print("PASS: row count = users × 4 weeks")

In [ ]:
# No duplicate (user_id, week_num) pairs
dup_count = weekly_features.groupBy("user_id", "week_num").count() \
    .filter(F.col("count") > 1).count()
assert dup_count == 0, f"Found {dup_count} duplicate (user_id, week_num) pairs"
print("PASS: no duplicate (user_id, week_num) pairs")

In [ ]:
# Week distribution should be uniform
week_counts = weekly_features.groupBy("week_num").count().orderBy("week_num")
week_counts.show()

counts = [row["count"] for row in week_counts.collect()]
assert len(counts) == 4, f"Expected 4 weeks, got {len(counts)}"
assert all(c == counts[0] for c in counts), f"Uneven week distribution: {counts}"
print("PASS: 4 weeks with uniform distribution")

In [ ]:
# Null inventory
null_counts = weekly_features.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in weekly_features.columns
]).collect()[0]

print("Null counts per column:")
for col_name in weekly_features.columns:
    n = null_counts[col_name]
    if n > 0:
        print(f"  {col_name}: {n:,}")

# Columns that should never be null
never_null = ["user_id", "week_num", "weekly_session_count",
              "avg_session_duration_min", "total_playtime_min",
              "weekly_session_count_norm", "total_playtime_min_norm",
              "peak_hour_ratio", "weekend_ratio", "session_regularity",
              "longest_inactive_days", "daily_playtime_std",
              "current_tier_numeric", "days_since_signup",
              "tier_changes_count", "has_downgraded",
              "total_spend_last_4w", "payment_count_last_4w",
              "failed_payment_count", "refund_count"]
for col_name in never_null:
    assert null_counts[col_name] == 0, f"{col_name} has {null_counts[col_name]} nulls"
print("PASS: zero-filled columns have no nulls")

# Count zero-session user-weeks (the root cause of most nulls)
zero_session_weeks = weekly_features.filter(F.col("weekly_session_count") == 0).count()
print(f"\nZero-session user-weeks: {zero_session_weeks:,} ({zero_session_weeks/200_000:.1%})")

# wow_change: week 1 always null + weeks 2-4 null only when current OR prior week has 0 sessions
for col_name in ["session_count_wow_change", "playtime_wow_change"]:
    week1_nulls = weekly_features.filter(
        (F.col("week_num") == 1) & F.col(col_name).isNull()
    ).count()
    assert week1_nulls == total_users, f"{col_name} should be null for all users in week 1"
print("PASS: wow_change null in all week-1 rows")

# Streaming features: nulls should exactly match zero-session weeks
streaming_cols = ["avg_latency", "avg_fps", "frame_drop_rate", "disconnect_rate",
                  "avg_bitrate", "avg_jitter", "packet_loss_avg", "crash_exit_ratio"]
for col_name in streaming_cols:
    assert null_counts[col_name] == zero_session_weeks, \
        f"{col_name} nulls ({null_counts[col_name]}) != zero-session weeks ({zero_session_weeks})"
print(f"PASS: streaming feature nulls match zero-session weeks ({zero_session_weeks:,})")

# Game features: nulls should match zero-session weeks
for col_name in ["unique_games_played", "top_game_concentration", "genre_entropy"]:
    assert null_counts[col_name] == zero_session_weeks, \
        f"{col_name} nulls ({null_counts[col_name]}) != zero-session weeks ({zero_session_weeks})"
print("PASS: game feature nulls match zero-session weeks")

## Level 2 — Value Range Sanity

In [ ]:
# --- Bounded [0, 1] features ---
bounded_cols = ["peak_hour_ratio", "weekend_ratio", "crash_exit_ratio",
                "packet_loss_avg", "top_game_concentration"]
for col_name in bounded_cols:
    if col_name not in weekly_features.columns:
        continue
    stats = weekly_features.select(
        F.min(col_name).alias("min_val"),
        F.max(col_name).alias("max_val"),
    ).collect()[0]
    assert stats["min_val"] >= 0.0, f"{col_name} min={stats['min_val']} < 0"
    assert stats["max_val"] <= 1.0, f"{col_name} max={stats['max_val']} > 1"
    print(f"  {col_name}: [{stats['min_val']:.4f}, {stats['max_val']:.4f}] ✓")

print("PASS: bounded features within [0, 1]")

In [ ]:
# --- Non-negative features ---
nonneg_cols = [
    "weekly_session_count", "avg_session_duration_min", "total_playtime_min",
    "weekly_session_count_norm", "total_playtime_min_norm",
    "session_regularity", "longest_inactive_days",
    "avg_latency", "avg_fps", "avg_bitrate", "avg_jitter",
    "frame_drop_rate", "disconnect_rate",
    "unique_games_played", "genre_entropy",
    "daily_playtime_std", "daily_playtime_cv", "session_duration_std",
    "days_since_signup", "tier_changes_count",
    "total_spend_last_4w", "payment_count_last_4w",
    "failed_payment_count", "refund_count",
]
for col_name in nonneg_cols:
    if col_name not in weekly_features.columns:
        continue
    min_val = weekly_features.select(F.min(col_name)).collect()[0][0]
    assert min_val is None or min_val >= 0, f"{col_name} min={min_val} is negative"

print("PASS: non-negative features have no negative values")

# --- Population normalization sanity check ---
# Normalized features should have population median ~1.0 per week (among active users)
for col_name in ["weekly_session_count_norm", "total_playtime_min_norm"]:
    medians = weekly_features.filter(F.col("weekly_session_count") > 0) \
        .groupBy("week_num").agg(
            F.expr(f"percentile_approx({col_name}, 0.5)").alias("median")
        ).orderBy("week_num").collect()
    values = [row["median"] for row in medians]
    print(f"  {col_name} active-user medians by week: {[f'{v:.2f}' for v in values]}")
    for v in values:
        assert 0.9 <= v <= 1.1, f"{col_name} median {v:.2f} deviates from 1.0"

print("PASS: normalized features have median ~1.0 per week (hot-week effect removed)")

In [ ]:
# --- Key percentile distributions (visual + soft checks) ---
print("Session Patterns:")
weekly_features.select(
    F.min("weekly_session_count").alias("min"),
    F.expr("percentile_approx(weekly_session_count, 0.5)").alias("p50"),
    F.max("weekly_session_count").alias("max"),
).show()

weekly_features.select(
    F.min("avg_session_duration_min").alias("min"),
    F.expr("percentile_approx(avg_session_duration_min, 0.5)").alias("p50"),
    F.expr("percentile_approx(avg_session_duration_min, 0.99)").alias("p99"),
    F.max("avg_session_duration_min").alias("max"),
).show()

print("Engagement Decay (weeks 2-4 only):")
weekly_features.filter(F.col("week_num") > 1).select(
    F.expr("percentile_approx(session_count_wow_change, 0.01)").alias("wow_p01"),
    F.expr("percentile_approx(session_count_wow_change, 0.5)").alias("wow_p50"),
    F.expr("percentile_approx(session_count_wow_change, 0.99)").alias("wow_p99"),
).show()

print("Streaming Quality:")
weekly_features.select(
    F.min("avg_latency").alias("lat_min"),
    F.expr("percentile_approx(avg_latency, 0.5)").alias("lat_p50"),
    F.max("avg_latency").alias("lat_max"),
    F.min("avg_fps").alias("fps_min"),
    F.expr("percentile_approx(avg_fps, 0.5)").alias("fps_p50"),
    F.max("avg_fps").alias("fps_max"),
).show()

## Level 3 — Persona Behavior Profiles

Validate that feature distributions match expected persona characteristics from `data_design.md`.

In [ ]:
persona_stats = weekly_features.join(
    users.select("user_id", "persona", "subscription_tier"), on="user_id"
)

persona_avgs = persona_stats.groupBy("persona").agg(
    F.avg("weekly_session_count").alias("avg_sessions"),
    F.avg("avg_session_duration_min").alias("avg_duration"),
    F.avg("total_playtime_min").alias("avg_playtime"),
    F.avg("peak_hour_ratio").alias("avg_peak"),
    F.avg("weekend_ratio").alias("avg_weekend"),
).collect()

pa = {row["persona"]: row for row in persona_avgs}

# Show the table
persona_stats.groupBy("persona").agg(
    F.avg("weekly_session_count").alias("avg_sessions"),
    F.avg("avg_session_duration_min").alias("avg_duration"),
    F.avg("total_playtime_min").alias("avg_playtime"),
    F.avg("peak_hour_ratio").alias("avg_peak"),
    F.avg("weekend_ratio").alias("avg_weekend"),
).show()

# Hardcore should have highest session count and playtime
assert pa["hardcore"]["avg_sessions"] > pa["regular"]["avg_sessions"], \
    "hardcore should have more sessions than regular"
assert pa["regular"]["avg_sessions"] > pa["casual"]["avg_sessions"], \
    "regular should have more sessions than casual"
assert pa["hardcore"]["avg_playtime"] > pa["regular"]["avg_playtime"], \
    "hardcore should have more playtime than regular"

print("PASS: session pattern persona ordering (hardcore > regular > casual)")

In [ ]:
# --- Engagement decay by persona ---
decay_avgs = persona_stats.groupBy("persona").agg(
    F.avg("session_count_wow_change").alias("avg_wow_change"),
    F.avg("session_count_vs_baseline").alias("avg_vs_baseline"),
    F.avg("playtime_vs_baseline").alias("avg_play_vs_base"),
    F.avg("longest_inactive_days").alias("avg_inactive"),
)
decay_avgs.show()

da = {row["persona"]: row for row in decay_avgs.collect()}

# about_to_churn should have lowest vs_baseline (declining relative to own history)
assert da["about_to_churn"]["avg_vs_baseline"] < da["regular"]["avg_vs_baseline"], \
    "about_to_churn should have lower vs_baseline than regular"
# about_to_churn should have longest inactive stretches
assert da["about_to_churn"]["avg_inactive"] > da["hardcore"]["avg_inactive"], \
    "about_to_churn should have longer inactive days than hardcore"

print("PASS: engagement decay persona ordering")

In [ ]:
# --- Game diversity by persona ---
game_avgs = persona_stats.groupBy("persona").agg(
    F.avg("unique_games_played").alias("avg_unique_games"),
    F.avg("genre_entropy").alias("avg_entropy"),
    F.avg("top_game_concentration").alias("avg_concentration"),
)
game_avgs.show()

ga = {row["persona"]: row for row in game_avgs.collect()}

# Hardcore plays most diverse games
assert ga["hardcore"]["avg_unique_games"] > ga["casual"]["avg_unique_games"], \
    "hardcore should play more unique games than casual"
assert ga["hardcore"]["avg_entropy"] > ga["casual"]["avg_entropy"], \
    "hardcore should have higher genre entropy than casual"
# Casual should have highest concentration (few games)
assert ga["casual"]["avg_concentration"] > ga["hardcore"]["avg_concentration"], \
    "casual should have higher top-game concentration than hardcore"

print("PASS: game diversity persona ordering")

In [ ]:
# --- Subscription & payment by persona ---
pay_avgs = persona_stats.groupBy("persona").agg(
    F.avg("current_tier_numeric").alias("avg_tier"),
    F.avg("total_spend_last_4w").alias("avg_spend"),
    F.avg("failed_payment_count").alias("avg_failed"),
    F.avg("refund_count").alias("avg_refund"),
    F.avg("payment_frequency_change").alias("avg_freq_change"),
    F.avg("has_downgraded").alias("downgrade_rate"),
)
pay_avgs.show()

pv = {row["persona"]: row for row in pay_avgs.collect()}

# Hardcore should have highest tier and spend
assert pv["hardcore"]["avg_tier"] > pv["casual"]["avg_tier"], \
    "hardcore should have higher avg tier than casual"
assert pv["hardcore"]["avg_spend"] > pv["casual"]["avg_spend"], \
    "hardcore should spend more than casual"
# about_to_churn should have highest downgrade rate
assert pv["about_to_churn"]["downgrade_rate"] > pv["hardcore"]["downgrade_rate"], \
    "about_to_churn should have higher downgrade rate than hardcore"

print("PASS: subscription & payment persona ordering")

## Level 4 — Churn Decay Progression

The most critical check: `about_to_churn` users should show progressive signal degradation
from week 1 to week 4. This is the core pattern the model must learn.

In [ ]:
churn_users = persona_stats.filter(F.col("persona") == "about_to_churn")

churn_weekly = churn_users.groupBy("week_num").agg(
    F.avg("weekly_session_count").alias("sessions_raw"),
    F.avg("weekly_session_count_norm").alias("sessions_norm"),
    F.avg("avg_session_duration_min").alias("duration"),
    F.avg("session_count_wow_change").alias("wow_change"),
    F.avg("playtime_vs_baseline").alias("vs_baseline"),
    F.avg("longest_inactive_days").alias("inactive_days"),
    F.avg("crash_exit_ratio").alias("crash_ratio"),
    F.avg("disconnect_rate").alias("disconnect_rate"),
).orderBy("week_num")

print("about_to_churn weekly trend:")
churn_weekly.show()

cw = {row["week_num"]: row for row in churn_weekly.collect()}

# Normalized session count should decrease monotonically (hot-week effect removed)
assert cw[4]["sessions_norm"] < cw[1]["sessions_norm"], \
    f"Churn users: week 4 norm sessions ({cw[4]['sessions_norm']:.2f}) should be < week 1 ({cw[1]['sessions_norm']:.2f})"
assert cw[3]["sessions_norm"] < cw[2]["sessions_norm"], \
    f"Churn users: week 3 norm sessions ({cw[3]['sessions_norm']:.2f}) should be < week 2 ({cw[2]['sessions_norm']:.2f})"

# vs_baseline should be below 1.0 by week 4 (declining vs own history)
assert cw[4]["vs_baseline"] < 1.0, \
    f"Churn users: week 4 vs_baseline ({cw[4]['vs_baseline']:.2f}) should be < 1.0"

# Inactive days should increase
assert cw[4]["inactive_days"] > cw[1]["inactive_days"], \
    f"Churn users: week 4 inactive days should be > week 1"

print("PASS: churn users show progressive decay (verified on normalized sessions)")

In [ ]:
# Contrast: hardcore should be flat/stable across all weeks
hardcore_users = persona_stats.filter(F.col("persona") == "hardcore")

hardcore_weekly = hardcore_users.groupBy("week_num").agg(
    F.avg("weekly_session_count").alias("sessions_raw"),
    F.avg("weekly_session_count_norm").alias("sessions_norm"),
    F.avg("avg_session_duration_min").alias("duration"),
    F.avg("crash_exit_ratio").alias("crash_ratio"),
    F.avg("longest_inactive_days").alias("inactive_days"),
).orderBy("week_num")

print("hardcore weekly trend:")
hardcore_weekly.show()

hw = {row["week_num"]: row for row in hardcore_weekly.collect()}

# Raw sessions may swing due to hot weeks, but normalized should be stable
raw_change = abs(hw[1]["sessions_raw"] - hw[4]["sessions_raw"]) / hw[1]["sessions_raw"]
norm_change = abs(hw[1]["sessions_norm"] - hw[4]["sessions_norm"]) / hw[1]["sessions_norm"]
print(f"  Raw session change week 1→4: {raw_change:.1%}")
print(f"  Normalized session change week 1→4: {norm_change:.1%}")

assert norm_change < 0.3, \
    f"Hardcore normalized sessions changed {norm_change:.0%} from week 1 to 4 — too much"

print(f"PASS: hardcore stable after normalization (norm change: {norm_change:.1%})")

## Summary

In [ ]:
print("="*50)
print("All validation checks passed!")
print("="*50)